# Model 1: 1D CNN Baseline
## Heart Murmur Detection - CirCor DigiScope Dataset
This notebook implements a 1D Convolutional Neural Network (CNN) as a baseline model 
for heart murmur detection using raw PCG (phonocardiogram) waveforms.

In [ ]:
# Imports and Setup
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Preprocessed Data
Loading the outputs from the data processing pipeline:
- `X_train` / `X_test` — raw waveforms (shape: num_heartbeats x 4000)
- `y_train` / `y_test` — labels (1 = murmur present, 0 = murmur absent)
- `train_patient_ids` / `test_patient_ids` — patient IDs for patient-level aggregation


In [ ]:
# Load preprocessed data 
X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
train_patient_ids = np.load("train_patient_ids.npy")
test_patient_ids = np.load("test_patient_ids.npy")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train — Murmur present: {y_train.sum()}, Absent: {(y_train==0).sum()}")
print(f"y_test  — Murmur present: {y_test.sum()}, Absent: {(y_test==0).sum()}")

## Normalization
We apply z-score normalization to each heartbeat independently.

**Why z-score?**
- Recordings come from different patients, stethoscopes, and volumes
- Z-score centers each signal around 0 and scales by standard deviation
- This removes loudness differences so the model learns actual patterns

**Formula:** `x_normalized = (x - mean) / std`

In [ ]:
def normalize_waveforms(X):
    """
    Apply z-score normalization to each heartbeat independently.
    Each heartbeat is normalized to have mean=0 and std=1.
    
    Args:
        X: numpy array of shape (num_heartbeats, 4000)
    Returns:
        X_normalized: numpy array of same shape
    """
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)
    
    # Avoid division by zero for silent recordings
    std = np.where(std == 0, 1, std)
    
    return (X - mean) / std

X_train_norm = normalize_waveforms(X_train)
X_test_norm = normalize_waveforms(X_test)

print(f"Before normalization — Mean: {X_train.mean():.2f}, Std: {X_train.std():.2f}")
print(f"After normalization  — Mean: {X_train_norm.mean():.2f}, Std: {X_train_norm.std():.2f}")

##  Class Weights
The dataset is imbalanced — roughly 79% no murmur, 21% murmur.
If we don't correct for this, the model will learn to always predict 
"no murmur" and appear accurate without actually detecting anything useful.

**Class weights** tell the model to penalize mistakes on the rare class 
(murmur present) more heavily during training.

In [ ]:
def compute_weights(y):
    """
    Compute class weights to handle class imbalance.
    
    Args:
        y: numpy array of labels (0 = absent, 1 = present)
    Returns:
        class_weights_tensor: torch tensor of weights for each class
    """
    classes = np.array([0, 1])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )
    
    print(f"Class weights — Absent: {weights[0]:.3f}, Present: {weights[1]:.3f}")
    class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
    return class_weights_tensor

class_weights = compute_weights(y_train)

## 1D CNN Architecture


### Architecture
```
Input (4000 samples)
        ↓
Conv1D + BatchNorm + ReLU + MaxPool    → detects low-level wave patterns
        ↓
Conv1D + BatchNorm + ReLU + MaxPool    → detects mid-level features
        ↓
Conv1D + BatchNorm + ReLU + MaxPool    → detects high-level features
        ↓
Global Average Pooling                 → summarizes features
        ↓
Fully Connected + Dropout              → learns classification
        ↓
Output (1 neuron → murmur probability)
```

In [5]:
class MurmurCNN1D(nn.Module):
    """
    1D CNN for heart murmur detection from raw PCG waveforms.
    
    Input:  (batch_size, 1, 4000) — raw waveform
    Output: (batch_size, 1)       — murmur probability
    """
    def __init__(self):
        super(MurmurCNN1D, self).__init__()
        
        # --- Convolutional Blocks ---
        # Each block: Conv1D → BatchNorm → ReLU → MaxPool
        
        # Block 1: detect low-level patterns (e.g. sharp peaks of S1/S2)
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)   # 4000 → 1000
        )
        
        # Block 2: detect mid-level features
        self.block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)   # 1000 → 250
        )
        
        # Block 3: detect high-level features
        self.block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)   # 250 → 62
        )
        
        # --- Global Average Pooling ---
        # Reduces each feature map to a single number
        # Makes model robust to waveform length variations
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # --- Classifier ---
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),      # randomly drops neurons to prevent overfitting
            nn.Linear(64, 1),
            nn.Sigmoid()            # outputs probability between 0 and 1
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x

# Initialize the model
model = MurmurCNN1D().to(device)
print(model)

MurmurCNN1D(
  (block1): Sequential(
    (0): Conv1d(1, 32, kernel_size=(15,), stride=(1,), padding=(7,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv1d(32, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (global_avg_pool): AdaptiveAvgPool1d(output_size=1)
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
   

## Dataset & DataLoader


Each item returns:
- `waveform` — normalized 1D signal of shape (1, 4000)
- `label`    — 0 (absent) or 1 (present)
- `patient_id` — for patient-level aggregation at evaluation

In [8]:
class HeartbeatDataset(Dataset):
    """
    PyTorch Dataset for heartbeat waveforms.
    
    Args:
        X:           numpy array of waveforms (num_heartbeats, 4000)
        y:           numpy array of labels (num_heartbeats,)
        patient_ids: numpy array of patient IDs (num_heartbeats,)
    """
    def __init__(self, X, y, patient_ids):
        # Add channel dimension: (num_heartbeats, 4000) → (num_heartbeats, 1, 4000)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = patient_ids
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]
